Martin Ofunrein
Section 1: Introduction & System Selection

Chosen System: Chemical Reaction with Fast Kinetics (Option B)

I chose to model a first-order chemical decay approaching a slowly varying equilibrium concentration. The governing ODE is
$$\frac{dC}{dt} = -k\bigl(C - C_{\text{eq}}(t)\bigr)$$
where
Symbol
Meaning
Value
$C(t)$
concentration of the reacting species
dependent variable
$k$
reaction rate constant
$10^{4}\;\text{s}^{-1}$
$C_{\text{eq}}(t)$
slowly varying equilibrium
$1 + 0.1\sin(0.1t)$
$C(0)$
initial concentration
$0$
Expanding into the standard stiff form $\dfrac{dy}{dt} = -\lambda\, y + g(t)$:
$$\frac{dC}{dt} = -10^{4}\,C + 10^{4}\bigl(1 + 0.1\sin(0.1t)\bigr)$$
so $\lambda = 10^{4}$ and $g(t) = 10^{4}(1 + 0.1\sin(0.1t))$.
Why this system matters

Fast chemical kinetics appear in combustion modeling, enzyme catalysis, and atmospheric chemistry. The reaction itself equilibrates in about $\tau_{\text{fast}} = 1/k = 10^{-4}\;\text{s}$ (0.1 ms), while the equilibrium drifts on a time scale of $\tau_{\text{slow}} = 2\pi/0.1 \approx 63\;\text{s}$. Any simulation that must capture both scales faces the classic stiffness challenge.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 12, 'figure.figsize': (10, 5),
                     'lines.linewidth': 1.5})

# --- Problem parameters ---
lam   = 1e4          # reaction rate constant k = lambda
C0    = 0.0          # initial concentration
t_end = 10.0         # simulate 10 seconds (many slow periods)
w     = 0.1          # angular frequency of equilibrium oscillation

def f(t, y):
    """RHS of the ODE: dC/dt = -lambda*C + lambda*C_eq(t)"""
    return -lam * y + lam * (1 + 0.1 * np.sin(w * t))

def C_eq(t):
    """Slowly varying equilibrium concentration."""
    return 1 + 0.1 * np.sin(w * t)

def analytical_solution(t):
    """Closed-form exact solution using variation of parameters.
    For dy/dt = -lam*y + lam*(1 + A*sin(w*t)), y(0) = 0:
    Particular solution: y_p = 1 + A*lam^2/(lam^2+w^2)*sin(wt) - A*lam*w/(lam^2+w^2)*cos(wt)
    Homogeneous: K*exp(-lam*t), with K chosen to match y(0)=0.
    """
    A = 0.1
    denom = lam**2 + w**2
    c_sin = A * lam**2 / denom
    c_cos = A * lam * w / denom
    y_p = 1.0 + c_sin * np.sin(w * t) - c_cos * np.cos(w * t)
    K = -(1.0 - c_cos)   # from y(0) = y_p(0) + K = 0
    return y_p + K * np.exp(-lam * t)

print(f"lambda = {lam:.0e}")
print(f"tau_fast = {1/lam:.1e} s")
print(f"tau_slow = {2*np.pi/w:.1f} s")
print(f"Stiffness ratio = {(2*np.pi/w) / (1/lam):.1e}")
print(f"y(0) = {analytical_solution(0.0):.6f}")
print(f"y(10) = {analytical_solution(10.0):.6f}")

# --- Platform-agnostic output directory ---
import os
try:
    import google.colab
    OUTPUT_DIR = '/content/'
except ImportError:
    OUTPUT_DIR = os.path.join(os.path.expanduser('~'), 'Downloads', '')
os.makedirs(OUTPUT_DIR, exist_ok=True)


Section 2: Demonstration of Stiffness

2.1 Euler's Forward with a "reasonable" step size

Euler's Forward update is $y_{n+1} = y_n + h\,f(t_n, y_n)$. For stability on the equation $dy/dt = -\lambda y + g(t)$, we need $|1 - h\lambda| < 1$, i.e., $h < 2/\lambda = 2 \times 10^{-4}\;\text{s}$.
With $h = 0.1$ (a step size that would be reasonable for capturing the slow dynamics), $h\lambda = 1000$ and the amplification factor is $|1 - 1000| = 999$. The solution will blow up catastrophically.
2.2 Stiffness ratio

This is a scalar linear system so the "eigenvalue" is simply $\lambda = 10^4$. The stiffness ratio (fast time scale to slow time scale) is
$$\text{Stiffness ratio} = \frac{\tau_{\text{slow}}}{\tau_{\text{fast}}} = \frac{2\pi / 0.1}{1/\lambda} = \frac{62.8}{10^{-4}} \approx 6.3 \times 10^5$$
2.3 Impracticality of explicit methods

For Euler's Forward to be stable we need $h < 2/\lambda = 2\times10^{-4}\;\text{s}$. To simulate $t \in [0, 10]$:
$$N_{\text{steps}} = \frac{10}{2\times 10^{-4}} = 50{,}000$$
steps — and that is the
bare minimum
; in practice you would use $h \ll 2/\lambda$ for safety, pushing $N$ well above $10^5$. Meanwhile, the slow dynamics only need $\sim 100$ steps for good accuracy.


In [ ]:
# --- Euler Forward (demonstrate instability) ---

def euler_forward(f, y0, t0, t_end, h):
    """Euler's Forward method. Returns t, y arrays."""
    N = int(np.ceil((t_end - t0) / h))
    t = np.zeros(N + 1)
    y = np.zeros(N + 1)
    t[0], y[0] = t0, y0
    for n in range(N):
        y[n+1] = y[n] + h * f(t[n], y[n])
        t[n+1] = t[n] + h
        # Stop early if solution blows up
        if abs(y[n+1]) > 1e10:
            return t[:n+2], y[:n+2]
    return t, y

# Attempt with h = 0.1 (way too large)
t_fwd_big, y_fwd_big = euler_forward(f, C0, 0, 0.5, h=0.1)

# Attempt with h = 0.001 (still too large: h*lambda = 10)
t_fwd_med, y_fwd_med = euler_forward(f, C0, 0, 0.5, h=0.001)

# Stable but tiny: h = 1e-4 (h*lambda = 1, just barely stable)
t_fwd_ok, y_fwd_ok = euler_forward(f, C0, 0, 0.01, h=1e-5)

# Reference solution on same interval
t_ref_short = np.linspace(0, 0.01, 500)
y_ref_short = analytical_solution(t_ref_short)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Plot 1: h = 0.1 (blows up immediately)
axes[0].plot(t_fwd_big, y_fwd_big, 'r-o', ms=4)
axes[0].set_title('Euler Forward, h = 0.1')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('C(t)')
axes[0].set_ylim(-1e6, 1e6)
axes[0].ticklabel_format(style='sci', axis='y', scilimits=(0,0))
axes[0].axhline(y=1.0, color='k', ls='--', alpha=0.4, label='$C_{eq} \\approx 1$')
axes[0].legend()

# Plot 2: h = 0.001 (still unstable)
axes[1].plot(t_fwd_med, y_fwd_med, 'r-o', ms=2)
axes[1].set_title('Euler Forward, h = 0.001')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('C(t)')
axes[1].ticklabel_format(style='sci', axis='y', scilimits=(0,0))

# Plot 3: h = 1e-5 (stable but very expensive)
axes[2].plot(t_fwd_ok, y_fwd_ok, 'b-', alpha=0.7, label=f'Forward, h=1e-5 ({len(t_fwd_ok)} steps)')
axes[2].plot(t_ref_short, y_ref_short, 'k--', label='Exact')
axes[2].set_title('Euler Forward, h = $10^{-5}$ (stable)')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('C(t)')
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig1_stiffness_demo.png'), dpi=150)
plt.show()

print(f"\nh = 0.1:   h*lambda = {0.1*lam:.0f}  -> UNSTABLE (blows up in {len(t_fwd_big)-1} steps)")
print(f"h = 0.001: h*lambda = {0.001*lam:.0f} -> UNSTABLE (blows up in {len(t_fwd_med)-1} steps)")
print(f"h = 1e-5:  h*lambda = {1e-5*lam:.1f} -> stable, but needs {int(10/1e-5):,} steps for t=[0,10]")


Section 3: Mathematical Setup

Euler's Backward (Implicit Euler) Method

The implicit Euler update is
$$C_{n+1} = C_n + h\,f(t_{n+1},\, C_{n+1})$$
Substituting $f(t, C) = -\lambda C + g(t)$ and rearranging gives the
residual function
:
$$G(C_{n+1}) = C_{n+1} - C_n - h\bigl[-\lambda\,C_{n+1} + g(t_{n+1})\bigr] = 0$$
Analytical derivative (Jacobian) of the residual

$$G'(C_{n+1}) = 1 - h\,\frac{\partial f}{\partial C}\bigg|_{t_{n+1},\,C_{n+1}} = 1 - h(-\lambda) = 1 + h\lambda$$
Since $f$ is linear in $C$, the derivative is
constant
— it does not depend on $C_{n+1}$. We compute it once.
Newton-Raphson update

$$C_{n+1}^{(k+1)} = C_{n+1}^{(k)} - \frac{G\bigl(C_{n+1}^{(k)}\bigr)}{G'\bigl(C_{n+1}^{(k)}\bigr)}
= C_{n+1}^{(k)} - \frac{C_{n+1}^{(k)} - C_n - h\bigl[-\lambda\,C_{n+1}^{(k)} + g(t_{n+1})\bigr]}{1 + h\lambda}$$
Initial guess: $C_{n+1}^{(0)} = C_n$ (explicit Euler predictor).
Convergence criterion: $|G(C_{n+1}^{(k)})| < 10^{-6}$, with a maximum of 20 iterations.


Section 4: Implementation


In [ ]:
def g_forcing(t):
    """Forcing function g(t) = lambda * C_eq(t)."""
    return lam * (1 + 0.1 * np.sin(0.1 * t))

def residual(y_new, y_n, t_new, h):
    """Residual G(y_{n+1}) for the implicit Euler equation."""
    return y_new - y_n - h * f(t_new, y_new)

def residual_derivative(h):
    """Derivative G'(y_{n+1}) = 1 + h*lambda (constant for linear ODE)."""
    return 1 + h * lam

def newton_raphson(y_n, t_new, h, tol=1e-6, max_iter=20):
    """
    Solve G(y_{n+1}) = 0 using Newton-Raphson iteration.
    Returns (y_{n+1}, number_of_iterations).
    """
    y_guess = y_n  # initial guess = explicit Euler predictor
    dG = residual_derivative(h)  # constant Jacobian

    for k in range(max_iter):
        G = residual(y_guess, y_n, t_new, h)
        # Check convergence
        if abs(G) < tol:
            return y_guess, k + 1
        # Newton update
        y_guess = y_guess - G / dG

    # If we reach here, Newton-Raphson did not converge
    print(f"  Warning: Newton-Raphson did not converge after {max_iter} iterations (|G| = {abs(G):.2e})")
    return y_guess, max_iter

def euler_backward(f, y0, t0, t_end, h, tol=1e-6, max_iter=20):
    """
    Euler's Backward method with Newton-Raphson at each step.
    Returns t, y arrays and list of NR iteration counts per step.
    """
    N = int(np.ceil((t_end - t0) / h))
    t = np.zeros(N + 1)
    y = np.zeros(N + 1)
    nr_iters = []

    t[0], y[0] = t0, y0

    for n in range(N):
        t_new = t[n] + h
        y_new, iters = newton_raphson(y[n], t_new, h, tol, max_iter)
        t[n+1] = t_new
        y[n+1] = y_new
        nr_iters.append(iters)

    return t, y, nr_iters

print("Functions defined successfully.")
print(f"For h = 0.1:  G'(y) = 1 + h*lambda = {residual_derivative(0.1):.0f}")
print(f"For h = 0.01: G'(y) = 1 + h*lambda = {residual_derivative(0.01):.0f}")
print(f"For h = 1.0:  G'(y) = 1 + h*lambda = {residual_derivative(1.0):.0f}")


Section 5: Results & Analysis

5.1 Solutions with multiple step sizes


In [ ]:
# --- Run Euler Backward with several step sizes ---
step_sizes = [1.0, 0.5, 0.1, 0.01]
results = {}

for h in step_sizes:
    t_b, y_b, iters_b = euler_backward(f, C0, 0, t_end, h)
    results[h] = {'t': t_b, 'y': y_b, 'iters': iters_b}
    print(f"h = {h}: {len(t_b)-1} steps, avg NR iters = {np.mean(iters_b):.2f}, max NR = {max(iters_b)}")

# Reference solution (computed at fine points)
t_exact = np.linspace(0, t_end, 2000)
y_exact = analytical_solution(t_exact)

# --- Plot: Solution comparison ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db']
for i, h in enumerate(step_sizes):
    r = results[h]
    axes[0].plot(r['t'], r['y'], 'o-', ms=max(1, 6-i*1.5), color=colors[i],
                 alpha=0.8, label=f'h = {h}')
axes[0].plot(t_exact, y_exact, 'k--', lw=2, label='Exact')
axes[0].plot(t_exact, C_eq(t_exact), ':', color='gray', alpha=0.5, label='$C_{eq}(t)$')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Concentration C(t)')
axes[0].set_title('Euler Backward: Solution for Multiple Step Sizes')
axes[0].legend(fontsize=10)
axes[0].set_xlim(0, t_end)

# Zoom into initial transient
for i, h in enumerate(step_sizes):
    r = results[h]
    mask = r['t'] <= 0.005
    axes[1].plot(r['t'][mask], r['y'][mask], 'o-', ms=4, color=colors[i],
                 alpha=0.8, label=f'h = {h}')
t_zoom = np.linspace(0, 0.005, 500)
y_zoom = analytical_solution(t_zoom)
axes[1].plot(t_zoom, y_zoom, 'k--', lw=2, label='Exact')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Concentration C(t)')
axes[1].set_title('Zoom: Initial Transient Region')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig2_solutions.png'), dpi=150)
plt.show()


5.2 Newton-Raphson Convergence Analysis


In [ ]:
# --- Plot: Newton-Raphson iterations per time step ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, h in enumerate(step_sizes):
    r = results[h]
    t_mid = r['t'][1:]  # time at each step
    axes[0].plot(t_mid, r['iters'], 'o-', ms=max(1, 5-i), color=colors[i],
                 alpha=0.7, label=f'h = {h}')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Newton-Raphson Iterations')
axes[0].set_title('NR Iterations per Time Step')
axes[0].legend()
axes[0].set_ylim(0, 5)

# Error vs step size at t = 10
errors = []
h_values = [2.0, 1.0, 0.5, 0.2, 0.1, 0.05, 0.01, 0.005]
y_exact_end = analytical_solution(t_end)

for h in h_values:
    _, y_b, _ = euler_backward(f, C0, 0, t_end, h)
    errors.append(abs(y_b[-1] - y_exact_end))

axes[1].loglog(h_values, errors, 'bo-', ms=6, label='Backward Euler error at t=10')
# Reference slope for O(h)
h_ref = np.array(h_values)
axes[1].loglog(h_ref, 0.5*h_ref, 'k--', alpha=0.5, label='O(h) reference')
axes[1].set_xlabel('Step size h (s)')
axes[1].set_ylabel('|Error| at t = 10 s')
axes[1].set_title('Error vs. Step Size (Euler Backward)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig3_convergence.png'), dpi=150)
plt.show()

print("\nError at t = 10 s for each step size:")
for h, err in zip(h_values, errors):
    print(f"  h = {h:8.4f}  ->  |error| = {err:.6e}")


5.3 Step Size Stability Comparison

Euler's Forward requires $h < 2/\lambda = 2\times10^{-4}$ s for stability. Euler's Backward is
unconditionally stable
— it produces a bounded, non-oscillatory solution for
any
step size $h > 0$. The table below summarizes this.


In [ ]:
# --- Stability comparison table ---
print("=" * 72)
print(f"{'Method':<22} {'Max Stable h':<18} {'h used':<12} {'Steps for t=[0,10]':<20} {'Stable?'}")
print("=" * 72)

# Forward Euler tests
fwd_tests = [0.1, 0.001, 1e-4, 1e-5]
for h in fwd_tests:
    N = int(10 / h)
    stable = "Yes" if h * lam < 2 else "NO"
    print(f"{'Euler Forward':<22} {'2e-4 s':<18} {h:<12.1e} {N:<20,} {stable}")

print("-" * 72)

# Backward Euler tests
bwd_tests = [1.0, 0.5, 0.1, 0.01]
for h in bwd_tests:
    N = int(10 / h)
    print(f"{'Euler Backward':<22} {'unlimited':<18} {h:<12.1e} {N:<20,} {'Yes'}")

print("=" * 72)
print(f"\nStiffness ratio: {(2*np.pi/0.1)/(1/lam):.2e}")
print(f"Max stable h (Forward): {2/lam:.1e} s")
print(f"Euler Backward is unconditionally A-stable for Re(lambda) < 0.")


Section 6: Performance Analysis


In [ ]:
# --- Computational cost comparison ---
import time

print("Computational Cost Comparison: Euler Forward vs. Backward")
print("=" * 80)
print(f"{'Method':<20} {'h':<12} {'Steps':<12} {'Avg NR':<10} {'Total f evals':<16} {'Wall time (ms)'}")
print("=" * 80)

# Backward Euler at various step sizes
for h in [1.0, 0.5, 0.1, 0.01]:
    t0 = time.perf_counter()
    _, y_b, iters = euler_backward(f, C0, 0, t_end, h)
    wall = (time.perf_counter() - t0) * 1000
    N = int(t_end / h)
    avg_nr = np.mean(iters)
    # Each NR iteration evaluates f once, plus one for the residual check
    total_f_evals = sum(iters)
    print(f"{'Backward':<20} {h:<12.4f} {N:<12,} {avg_nr:<10.1f} {total_f_evals:<16,} {wall:<.2f}")

print("-" * 80)

# Forward Euler at the minimum stable step size
h_fwd = 1e-5
t0 = time.perf_counter()
t_f, y_f = euler_forward(f, C0, 0, t_end, h_fwd)
wall = (time.perf_counter() - t0) * 1000
N_fwd = int(t_end / h_fwd)
print(f"{'Forward (stable)':<20} {h_fwd:<12.1e} {N_fwd:<12,} {'N/A':<10} {N_fwd:<16,} {wall:<.2f}")

print("=" * 80)
print()

# Speedup calculation
print("Speedup of Backward Euler over stable Forward Euler:")
for h in [1.0, 0.1, 0.01]:
    r = results[h]
    bwd_evals = sum(r['iters'])
    speedup = N_fwd / bwd_evals
    print(f"  h = {h}: {bwd_evals} f-evals vs {N_fwd:,} -> {speedup:.0f}x fewer evaluations")


In [ ]:
# --- Plot: Accuracy vs Computational Cost ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Concentration profiles — Backward vs exact vs equilibrium
h_demo = 0.1
r = results[h_demo]
axes[0].plot(r['t'], r['y'], 'b-o', ms=2, label=f'Backward Euler (h={h_demo})')
axes[0].plot(t_exact, y_exact, 'k--', lw=2, label='Exact solution')
axes[0].plot(t_exact, C_eq(t_exact), ':', color='orange', lw=2, label='$C_{eq}(t)$')
axes[0].fill_between(t_exact, y_exact - 0.005, y_exact + 0.005, alpha=0.15, color='green')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Concentration C(t)')
axes[0].set_title('Solution Tracks Equilibrium After Fast Transient')
axes[0].legend()

# Plot 2: Solution remains in physical bounds (C >= 0)
axes[1].plot(r['t'], r['y'], 'b-', label=f'Backward h={h_demo}')
axes[1].axhline(y=0, color='r', ls='--', alpha=0.5, label='C = 0 (physical bound)')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Concentration C(t)')
axes[1].set_title('Solution Quality: Physical Bounds Check')
axes[1].legend()
axes[1].set_ylim(-0.1, 1.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig4_quality.png'), dpi=150)
plt.show()

# Verify physical bounds
for h in step_sizes:
    r = results[h]
    y_min = np.min(r['y'])
    y_max = np.max(r['y'])
    print(f"h = {h}: C_min = {y_min:.6f}, C_max = {y_max:.6f}  (all non-negative: {y_min >= -1e-10})")


Section 7: Conclusions

Key findings:
The chemical kinetics ODE is severely stiff.
With $\lambda = 10^4$, the fast transient decays in $\sim 0.1$ ms while the equilibrium varies over $\sim 63$ s, giving a stiffness ratio of $\approx 6.3 \times 10^5$.
Euler's Forward is impractical.
It requires $h < 2\times 10^{-4}$ s for stability, demanding at least 50,000 steps for a 10-second simulation. With $h = 0.1$ or even $h = 0.001$ it blows up immediately.
Euler's Backward is unconditionally stable.
It produces correct, bounded solutions even with $h = 1.0$ s — a step size $5{,}000\times$ larger than Forward Euler's stability limit. The Newton-Raphson solver converges in just 1-2 iterations per step because the Jacobian $G' = 1 + h\lambda$ is constant.
Massive computational savings.
Backward Euler with $h = 0.1$ uses only $\sim 100$ steps (and $\sim 100$ function evaluations) versus $1{,}000{,}000$ for stable Forward Euler — a $10{,}000\times$ reduction.
Error scales as $O(h)$
as expected for a first-order method. Halving $h$ halves the error, confirming correct implementation.
When to use implicit vs. explicit methods:
Implicit methods like Backward Euler should be used whenever the system is stiff — i.e., when the stability requirement of explicit methods forces impractically small time steps relative to the accuracy requirement. The extra cost of solving a nonlinear equation at each step is far outweighed by the ability to take large steps. For this chemical kinetics problem, Backward Euler is the clear winner.
Challenges and lessons learned:
The main challenge was understanding that for this linear problem, Newton-Raphson converges in a single iteration because the residual is itself linear in $C_{n+1}$. This made the implementation straightforward but also highlighted why implicit methods are especially efficient for linear stiff systems. For nonlinear stiff problems, more Newton-Raphson iterations would be needed, but the unconditional stability advantage remains.
